# RF Root Cause SLM — QLoRA fine-tune on a Colab T4
Runtime > Change runtime type > **T4 GPU** first. Run cells top to bottom.
Uses `training/qlora_config.t4.yaml` (Qwen2.5-1.5B-Instruct, 4-bit, fp16, batch 1 x accum 32, seq 1536).
Checkpoints go to Google Drive so a disconnect does not lose the adapter.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: switch the runtime to T4"

In [ ]:
!git clone https://github.com/natrajexplore/SLM_Building_Wi-Fi_RF_issues.git rf-slm
%cd rf-slm
# trl>=1.0 / transformers>=5 are what train.py is written against; Colab's preinstalled ones are older.
!pip install -q -r requirements.txt
!pip install -q -U trl transformers peft accelerate bitsandbytes datasets

## Data
`data/train.jsonl` and `data/eval.jsonl` are git-ignored, so upload `data.zip` (made on your PC at `.local/colab/data.zip`).

In [ ]:
from google.colab import files
import zipfile
up = files.upload()  # pick data.zip
zipfile.ZipFile(next(iter(up))).extractall("data")
!ls -la data/*.jsonl

In [ ]:
# Keep training/out on Drive so checkpoints survive a disconnect.
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/rf-slm-out
!rm -rf training/out && ln -s /content/drive/MyDrive/rf-slm-out training/out

In [ ]:
!python -m training.train --dry-run --config training/qlora_config.t4.yaml | tail -3

## Train (about 220 optimizer steps; expect a few hours on a T4)

In [ ]:
!python -m training.train --config training/qlora_config.t4.yaml

## Score the adapter against the eval targets

In [ ]:
!python -m training.evaluate --responder adapter --model training/out

## Get the adapter back
The adapter is already in Drive at `MyDrive/rf-slm-out`. Download it, put it at `training/out/` in the repo, then run the backend with `RF_SLM_BACKEND=adapter`.

In [ ]:
!cd /content/drive/MyDrive && zip -qr /content/rf-slm-adapter.zip rf-slm-out -x 'rf-slm-out/checkpoint-*'
from google.colab import files; files.download("/content/rf-slm-adapter.zip")